In [ ]:
def deriv(h,m2,m1,p1,p2):

    return (m2-8*m1+8*p1-p2)/(12.0*h) ###5 point stencil

def LF_MegaMapper(z,m):

    if z <= 2.5: #z=2

        Mstaruv, phi_star, alpha, muv = [-20.60,9.7,-1.6,24.2]

    elif z > 2.5 and z <=3.5 : #z=3

        Mstaruv, phi_star, alpha, muv = [-20.86,5.04,-1.78,24.7]

    elif z > 3.5 and z <=4.5 : #z=3.8

        Mstaruv, phi_star, alpha, muv = [-20.63,9.25,-1.57,25.4]

    elif z > 4.5 and z <=5.5 : #z=4.9

        Mstaruv, phi_star, alpha, muv = [-20.96,3.22,-1.6,25.5]

    elif z > 5.5 and z <=6 : #z=5.9

        Mstaruv, phi_star, alpha, muv = [-20.91,1.64,-1.87,25.8]

    #phi_star Mpc^3

    return np.log(10)/2.5*(phi_star/(10*0.67))*10**(-0.4*(1+alpha)*(m-Mstaruv))*np.exp(-10**(-0.4*(m-Mstaruv)))



def M_cutoff(z,m):

    return m - 5*np.log10(d_L(z)*1e6/10)+2.5*np.log10(1+z)



def NL_M_MP_MegaMap(z,M_cut):

    if len(z)>0:

        NL_res=np.zeros(len(z))

        for i in range(len(z)):

            NL_res[i]=quad(lambda m: LF_MegaMapper(z[i],m),-100,M_cut[i])[0]



    else:

        NL_res=quad(lambda m: LF_MegaMapper(z,m),-100,M_cut)[0]

    return NL_res



def make_biases_Mega():#z_range = np.arange(0.001,3.02,0.01)):

    z_range = np.arange(2.001,5.999,0.001)

    h_m=0.1

    m_cut_int=24.5+h_m*np.array([-2,-1,1,2])

    M_cut_der=np.array([M_cutoff(z_range,m) for m in m_cut_int])



    logNL_full_m2=np.log10(NL_M_MP_MegaMap(z_range,M_cut_der[0])*r(z_range)**2/H(z_range)*c_speed)

    logNL_full_m1=np.log10(NL_M_MP_MegaMap(z_range,M_cut_der[1])*r(z_range)**2/H(z_range)*c_speed)

    logNL_full_p1=np.log10(NL_M_MP_MegaMap(z_range,M_cut_der[2])*r(z_range)**2/H(z_range)*c_speed)

    logNL_full_p2=np.log10(NL_M_MP_MegaMap(z_range,M_cut_der[3])*r(z_range)**2/H(z_range)*c_speed)

    s_full_raw=deriv(h_m,logNL_full_m2,logNL_full_m1,logNL_full_p1,logNL_full_p2)

    s_int = scipy.interpolate.interp1d(z_range,s_full_raw)



    N = - (1+z_range)*np.gradient(np.log(NL_M_MP_MegaMap(z_range,M_cutoff(z_range,24.5))*r(z_range)**2/H(z_range)*c_speed))

    be_int = scipy.interpolate.interp1d(z_range,N)



    return s_int, be_int
